# Ensamble final Analista Jr

Promedia las probabilidades de las corridas del workflow `z728_final_junior` modificado
(cada corrida ya es un semillerío de 100 modelos LightGBM), ordena a los clientes de 202109
y genera los archivos para Kaggle.

Correr con el runtime en lenguaje **R**, después de que terminaron todas las corridas.

In [11]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Nov 15 17:09:32 2026"

In [12]:
rm(list=ls(all.names=TRUE))
gc(full=TRUE, verbose=FALSE)

require("data.table")
if( !require("yaml")) install.packages("yaml")
require("yaml")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,752660,40.2,1479647,79.1,1479647,79.1
Vcells,1417636,10.9,8388608,64.0,4292768,32.8


#### Parametros

In [13]:
PARAM <- list()
PARAM$experimento <- 9592

# corridas que entran al ensamble, todas con el mismo peso
#  9510 y 9511: sin los meses de pandemia 202003 a 202007
#  9520: con todos los meses
PARAM$ensamble$corridas <- c(9510, 9511, 9520, 9530)

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"
PARAM$kaggle$cortes <- seq(1400, 2600, by = 100)

#### Carpeta del Experimento

In [14]:
setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

#### Lectura de las predicciones de cada corrida

In [15]:
# cada corrida grabo WF<experimento>/prediccion.txt
#  con numero_de_cliente y prob = promedio de sus 100 modelos
leer_corrida <- function( exp ) {
  arch <- paste0("/content/buckets/b1/exp/WF", exp, "/prediccion.txt")
  if( !file.exists(arch) ) stop("No existe ", arch)
  tb <- fread(arch, select= c("numero_de_cliente", "prob"))
  setnames(tb, "prob", paste0("prob_", exp))
  tb
}

tbs <- lapply(PARAM$ensamble$corridas, leer_corrida)
print(sapply(tbs, nrow))  # las corridas tienen que tener la misma cantidad de clientes

tb_ensamble <- Reduce(function(a, b) merge(a, b, by= "numero_de_cliente"), tbs)
nrow(tb_ensamble)  # tiene que coincidir con el valor de arriba

[1] 33080 33080 33080 33080


[1] 33080

In [16]:
# que tan parecidas son las corridas
cols_prob <- paste0("prob_", PARAM$ensamble$corridas)
round(cor(tb_ensamble[, cols_prob, with= FALSE]), 3)

,prob_9510,prob_9511,prob_9520,prob_9530
prob_9510,1.000,0.995,0.993,0.970
prob_9511,0.995,1.000,0.989,0.962
prob_9520,0.993,0.989,1.000,0.966
prob_9530,0.970,0.962,0.966,1.000


#### Ensamble: promedio simple de las probabilidades

In [17]:
tb_ensamble[, prob := rowMeans(.SD), .SDcols= cols_prob]

fwrite(tb_ensamble,
  file= "prediccion_ensamble.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

In [18]:
tb_prediccion <- tb_ensamble[, list(numero_de_cliente, prob)]

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento, "_",
    "ens", length(PARAM$ensamble$corridas), "_",
    envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
    "  ensamble=", paste(PARAM$ensamble$corridas, collapse= "+"),
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [19]:
# grabo los parametros
write_yaml( PARAM, file="PARAM.yml")

In [20]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Nov 15 17:16:32 2026"